# <center>Code métriques 3STR</center>

Quels sont les buts de ce notebook ?  
- Mettre en forme le code sur les métriques pour le 3STR (prends en entrée une TS, une TS reconstruite et une liste d'index des dates nuageuses.  
      - Sortir des métriques occluded / observed / oa  
      - Organiser moi même comment les métriques sont faites vis-àvis des dates etcs  
- Prendre mieux en mains le code d'inférence d'U-TILISE  
      - Faire de premiers tests  
      - Voir si le système d'imputation peut-avoir un batch_size plus grand que 1   

### Inférence (reprise script eval)
Utilisation du script d'imputation pour avoir les prédictions (la ts reconstruite)

In [1]:
import collections.abc
import logging
import random
import re
from functools import partial
from typing import Any
from typing import Dict
from typing import List
from typing import Optional
from typing import Tuple
from typing import Union
from pathlib import Path
import numpy as np
import torch
import torch.utils
import torch.utils.data
from omegaconf import DictConfig
from torch import Tensor
from torch.nn import functional as F
from torch.utils.data import Dataset
from dataloader_CIRCA.datasets import CIRCA_ADAPTED2UTILISE_Dataset

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # ou FutureWarning, UserWarning, ...

In [2]:
# Définition du dataset
SUBSET_LENGTH = 20
batch_size_inference = 1

filter_settings = {
    "type": "cloud-free",  # Strategy for removing observations with data gaps.
    # ['cloud-free', 'cloud-free_consecutive']
    "min_length": 5,  # Minimum sequence length.
    "return_valid_obs_only": True,  # True to return the cloud-filtered sequences, False otherwise.
    # "max_t_sampling": 10,            # Maximum temporal sampling frequency in days.
}

mask_kwargs = {
    "mask_type": "random_clouds",  # Mask the input time series with randomly sampled cloud masks or the actual cloud masks. ['random_clouds', 'real_clouds']
    "ratio_masked_frames": 0.5,  # Ratio of partially/fully masked images per image time series (upper bound).
    "ratio_fully_masked_frames": 0.0,  # Ratio of fully masked images per image time series (upper bound).
    "fixed_masking_ratio": False,  # True to vary the masking ratio across different image time series, False otherwise.
    "non_masked_frames": [
        0
    ],  # list of int, time steps to be excluded from masking. E.g., [0] never masks the first frame in a sequence.
    "intersect_real_cloud_masks": False,  # True to intersect randomly sampled cloud masks with the actual cloud masks, False otherwise.
    "dilate_cloud_masks": False,  # True to dilate the cloud masks before masking, False otherwise.
    "fill_type": "fill_value",  # Strategy for initializing masked pixels. ['fill_value', 'white_noise', 'mean']
    "fill_value": 1,  # Pixel value of masked pixels. Used if fill_type == 'fill_value'.
    "p_filter": 0.1,
}

params_dataset = {
    'phase': "test",
    'hdf5_file': "/DATA_10TB/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5",
    'shuffle': False,
    'use_sar': 'mix_closest',
    'channels': "all",
    # U-TILISE specific parameters
    'filter_settings': filter_settings,
    'max_seq_length': 30,
    'render_occluded_above_p': None, # Set to None to keep original cloud masks. Minimum cloud cover to fully mask an input image (0.9 demo config)
    'mask_kwargs': mask_kwargs,
    'pe_strategy': "day-within-sequence",
    'augment': False,
    'process_data': True,
    'seed': 42,
    # Récupération de vieux arguments du repo
    'crop_settings': None,
    'return_cloud_mask': True,
}

dset = CIRCA_ADAPTED2UTILISE_Dataset(**params_dataset)

In [3]:
from lib.data_utils import seed_worker, pad_collate
from lib import config_utils
# dset = torch.utils.data.Subset(dset, range(SUBSET_LENGTH))
dataloader = torch.utils.data.DataLoader(
    dataset=dset,
    batch_size=batch_size_inference,
    shuffle=False,
    num_workers=0,
    collate_fn=None,
    pin_memory=False,
    drop_last=False,
)

### Chargement du modèle

In [4]:
from lib.models.utilise import UTILISE
from lib.eval_tools import impute_sequence
from lib import data_utils
temporal_window = dset.max_seq_length
num_channels = dset.num_channels
device = torch.device("cuda:0")
path_trainings_results = Path("/DATA_10TB/data_rpg/outputs/U-TILISE/results/ALL_SAR_120_epochs_2025-07-11_16-56")
path_ckpt = path_trainings_results / "checkpoints" / "Model_best.pth" 
path_config_training = path_trainings_results / "config.yaml" 
assert path_ckpt.exists()
assert path_config_training.exists()
config_training = config_utils.read_config(path_config_training)
config_training.utilise.input_dim = num_channels
config_training.utilise.output_dim = 10 # num_channels - 4 if use_sar
model = UTILISE(**config_training.utilise)
checkpoint = torch.load(path_ckpt)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device).eval()
del checkpoint

### Inférence sur un batch

In [5]:
def infer_one_batch(batch, model, temporal_window, device, t_start=None, t_end=None):
    if t_start is not None and t_end is not None:
        # Choose a subsequence
        batch["x"] = batch["x"][:, t_start:t_end, ...]
    
        for key in ["y", "masks", "cloud_mask", "masks_valid_obs"]:
            if key in batch:
                batch[key] = batch[key][:, t_start:t_end, ...]
    
        for key in ["days", "position_days"]:
            if key in batch:
                batch[key] = batch[key][:, t_start:t_end]
    
    batch = data_utils.to_device(batch, device)
    y_pred = impute_sequence(model, batch, temporal_window, return_att=False)
    batch = data_utils.to_device(batch, "cpu")
    y_pred = y_pred.cpu()
    return batch, y_pred

In [6]:
batch = next(iter(dataloader))

In [7]:
batch['y'].shape

torch.Size([1, 29, 10, 256, 256])

In [8]:
batch_processed, y_pred = infer_one_batch(
    batch=batch, 
    model=model, 
    temporal_window=temporal_window, 
    device=device, 
    t_start=0, 
    t_end=10,
)

In [9]:
batch_processed['y'].shape

torch.Size([1, 10, 10, 256, 256])

In [10]:
y_pred.shape

torch.Size([1, 10, 10, 256, 256])

### Code métriques

In [11]:
import math
from typing import Any
from typing import Dict
from typing import Literal

import torch
import torchgeometry as tgm
from prodict import Prodict
from torch import Tensor

In [12]:
class EvalMetrics:
    """
    Computes the metrics used to monitor the training progress or for evaluation.
    """
    def __init__(
        self, 
        masked_metrics: bool = False,
        sam_units: str = "rad", # "deg" or "rad"
        eval_occluded_observed: bool = True,  #  to evaluate the metrics over all pixels and separately for occluded and observed input pixels 
        mae: bool = True,
        rmse: bool = True,
        mse: bool = True,
        psnr: bool = True,
        sam: bool = True,
        ssim: bool = True,
    ):
        self.masked_metrics = masked_metrics
        # True to evaluate the metrics over all pixels and separately for occluded and observed input pixels;
        # False to evaluate the metrics over all pixels only
        self.eval_occluded_observed = eval_occluded_observed

        # MAE (mean absolute error)
        if mae:
            self.mae = lambda predicted, target: torch.mean(torch.abs(predicted - target))
        # MSE (mean squared error)
        if mse:
            self.mse = lambda predicted, target: torch.mean(torch.square(predicted - target))
        # RMSE (root mean square error)
        if rmse:
            self.rmse = lambda predicted, target: torch.sqrt(torch.mean(torch.square(predicted - target)))
        # SSIM (structural similarity index)
        if ssim:
            self.dssim = tgm.losses.SSIM(5, reduction="mean")
        # PSNR (peak signal-to-noise ratio)
        if psnr:
            self.psnr = lambda predicted, target: 20 * torch.log10(1 / self.rmse(predicted, target))
        # SAM (spectral angle mapper)
        if sam:
            self.sam_units = sam_units
            self.sam = EvalMetrics.compute_sam
    
    @staticmethod
    def compute_sam(predicted: Tensor, target: Tensor, units: Literal["deg", "rad"] = "rad") -> Tensor:
        """
        Computes the spectral angle mapper (SAM) averaged over all time steps and batch samples.
    
        Args:
            predicted:   torch.Tensor,  (n_frames x C x H x W).
            target:      torch.Tensor,  (n_frames x C x H x W).
    
        Returns:
            sam_value:   torch.Tensor, (1, ), mean spectral angle [rad].
        """
        dot_product = (predicted * target).sum(dim=1)
        predicted_norm = predicted.norm(dim=1)
        target_norm = target.norm(dim=1)
        # Compute the SAM score for all pixels with vector norm > 0
        flag = torch.logical_and(predicted_norm != 0.0, target_norm != 0.0)
        if torch.any(flag):
            spectral_angles = torch.clamp(dot_product[flag] / (predicted_norm[flag] * target_norm[flag]), -1, 1).acos()
            sam_score = torch.mean(spectral_angles)
            if units == "deg":
                sam_score *= 180 / math.pi
            return sam_score
        else:
            return None

    def __call__(self,
                 target: Tensor, 
                 masks: Tensor, 
                 predicted: Tensor) -> Dict[str, float]:
        """
        Args: 
            target:       torch.Tensor, (B x T x C x W x H); target sequence.
            masks:        torch.Tensor, (B x T x 1 x W x H); a pixel value of 0 indicates an
                          observed (non-masked) input pixel and a pixel value of 1 a masked input
                          pixel.
            cloud_mask:   torch.Tensor, (B x T x 1 x W x H), 0 indicates a non-occluded target pixel
                          and 1 an occluded target pixel.
            predicted:       torch.Tensor, (B x T x C x W x H); predicted sequence.
        """
        # Initialize metrics
        metrics = dict()

        # Concatenate batch and time dimension
        B, T, C, H, W = predicted.shape
        n_frames = B * T
        predicted = predicted.view(n_frames, C, H, W)
        target = target.view(n_frames, C, H, W)
        masks = masks.view(n_frames, 1, H, W).expand(target.shape)

        # Structural similarity index (SSIM) evaluated over all images
        if hasattr(self, "dssim"):
            dssim = self.dssim(predicted, target)  # outputs (1 - SSIM)/2; structural dissimilarity
            metrics["ssim"] = 1 - 2 * dssim

            # Structural similarity index (SSIM) evaluated over all images with data gaps
            if self.eval_occluded_observed:
                occ_images = (masks == 1.0).any(dim=-1).any(dim=-1).any(dim=-1)
                metrics["ssim_images_occluded_input_pixels"] = 1 - 2 * self.dssim(predicted[occ_images], target[occ_images])
                metrics["ssim_images_observed_input_pixels"] = 1 - 2 * self.dssim(predicted[~occ_images], target[~occ_images])

        # if self.masked_metrics == False: metrics are computed over all output pixels
        # if self.masked_metrics == True: metrics are computed over all non-occluded target pixels (according to GT cloud masks)
        # if self.masked_metrics:
        #     cloud_mask = cloud_mask.view(n_frames, 1, H, W)

        #     # Evaluate non-occluded target pixels only
        #     flag = cloud_mask.permute(0, 2, 3, 1).reshape(n_frames * H * W) == 0.0
        #     # print(flag.shape)
        #     # Tensor shapes: (n_frames * H * W, C)
        #     predicted = predicted.permute(0, 2, 3, 1).reshape(n_frames * H * W, C)[flag]
        #     target = target.permute(0, 2, 3, 1).reshape(n_frames * H * W, C)[flag]
        #     masks = masks.permute(0, 2, 3, 1).reshape(n_frames * H * W, C)[flag]

        # MAE (mean absolute error) evaluated over all pixels in the input sequence
        if hasattr(self, "mae"):
            metrics[f"mae"] = self.mae(predicted, target)
            if self.eval_occluded_observed:
                metrics[f"mae_occluded_input_pixels"] = self.mae(predicted[masks == 1.0], target[masks == 1.0])
                metrics[f"mae_observed_input_pixels"] = self.mae(predicted[masks == 0.0], target[masks == 0.0])

        # Root mean squared error (RMSE)
        if hasattr(self, "rmse"):
            metrics[f"rmse"] = self.rmse(predicted, target)
            if self.eval_occluded_observed:
                metrics[f"rmse_occluded_input_pixels"] = self.rmse(predicted[masks == 1.0], target[masks == 1.0])
                metrics[f"rmse_observed_input_pixels"] = self.rmse(predicted[masks == 0.0], target[masks == 0.0])

        # Mean squared error (MSE)
        if hasattr(self, "mse"):
            metrics[f"mse"] = self.mse(predicted, target)
            if self.eval_occluded_observed:
                metrics[f"mse_occluded_input_pixels"] = self.mse(predicted[masks == 1.0], target[masks == 1.0])
                metrics[f"mse_observed_input_pixels"] = self.mse(predicted[masks == 0.0], target[masks == 0.0])

        # PSNR
        if hasattr(self, "psnr"):
            metrics[f"psnr"] = self.psnr(predicted, target)
            if self.eval_occluded_observed:
                metrics[f"psnr_occluded_input_pixels"] = self.psnr(predicted[masks == 1.0], target[masks == 1.0])
                metrics[f"psnr_observed_input_pixels"] = self.psnr(predicted[masks == 0.0], target[masks == 0.0])

        # SAM
        if hasattr(self, "sam"):
            metrics[f"sam"] = self.compute_sam(predicted, target, units=self.sam_units)
            if self.eval_occluded_observed:
                metrics[f"sam_occluded_input_pixels"] = self.sam(predicted[:, :, (masks == 1.0).all(dim=1), :], target[:, :, (masks == 1.0).all(dim=1), :], units=self.sam_units)
                metrics[f"sam_observed_input_pixels"] = self.sam(predicted[:, :, (masks == 0.0).all(dim=1), :], target[:, :, (masks == 0.0).all(dim=1), :], units=self.sam_units)

        for key, value in metrics.items():
            metrics[key] = value.item()

        return metrics

In [13]:
compute_metrics = EvalMetrics(sam=False)

metrics_on_sample = compute_metrics(
    target=batch["y"], 
    masks=batch["masks"], 
    predicted=y_pred)

print("**Metrics on the sample:**")
for k, v in metrics_on_sample.items():
    print(f"{k}: {v:.4f}")

**Metrics on the sample:**
ssim: 0.9078
ssim_images_occluded_input_pixels: 0.8397
ssim_images_observed_input_pixels: 0.9759
mae: 0.0604
mae_occluded_input_pixels: 0.1311
mae_observed_input_pixels: 0.0106
rmse: 0.1323
rmse_occluded_input_pixels: 0.2049
rmse_observed_input_pixels: 0.0160
mse: 0.0175
mse_occluded_input_pixels: 0.0420
mse_observed_input_pixels: 0.0003
psnr: 17.5675
psnr_occluded_input_pixels: 13.7703
psnr_observed_input_pixels: 35.8972


In [90]:
masks=batch["masks"]
target=batch["y"]
predicted=y_pred

print(masks.shape)
# print(predicted.shape)

B, T, C, H, W = predicted.shape
n_frames = B * T
predicted = predicted.view(n_frames, C, H, W)
# target = target.view(n_frames, C, H, W)
masks = masks.view(n_frames, 1, H, W)
        

print(masks.shape)
# masks = masks.expand(target.shape)

print(masks.shape)
# print(predicted.shape)

torch.Size([1, 10, 1, 256, 256])
torch.Size([10, 1, 256, 256])
torch.Size([10, 1, 256, 256])
